# Exercises - Lesson 1.4: Math Foundations 4 - Transformer Plumbing

In [ ]:
!pip install -q numpy torch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Skill 1 - LayerNorm + RMSNorm

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
ln = nn.LayerNorm(4)
rms = nn.RMSNorm(4)
print(f'LayerNorm: {ln(x).round(decimals=3)}')
print(f'  mean: {ln(x).mean().item():.3f}, var: {ln(x).var(unbiased=False).item():.3f}')
print(f'RMSNorm:   {rms(x).round(decimals=3)}')
print(f'  mean: {rms(x).mean().item():.3f} (not zero), RMS: 1.0')

## Skill 2 - Sinusoidal positional encoding

In [ ]:
def sinusoidal_pe(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    pos = torch.arange(0, seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = sinusoidal_pe(seq_len=8, d_model=16)
print(f'PE shape: {pe.shape}')
print(f'PE[0]: {pe[0][:4].round(decimals=3)}')
print(f'PE[7]: {pe[7][:4].round(decimals=3)}')

## Skill 3 - Residual connections

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, d, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.layer = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        f = self.layer(x)
        return x + f if self.use_residual else f

def grad_at_input(use_residual, depth=20):
    torch.manual_seed(0)
    blocks = nn.Sequential(*[ResidualBlock(8, use_residual) for _ in range(depth)])
    x = torch.randn(1, 8, requires_grad=True)
    y = blocks(x).sum(); y.backward()
    return x.grad.norm().item()

print(f'20 layers, sigmoid, NO residual: grad norm = {grad_at_input(False):.4e}')
print(f'20 layers, sigmoid, WITH residual: grad norm = {grad_at_input(True):.4e}')

## The full Pre-LN transformer block (production pattern)

In [ ]:
class PreLNBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=8):
        super().__init__()
        self.norm1 = nn.RMSNorm(d_model)
        self.norm2 = nn.RMSNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
    def forward(self, x):
        B, T, D = x.shape
        h = self.norm1(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        a = a.transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.out(a)
        x = x + self.ffn(self.norm2(x))
        return x

block = PreLNBlock(d_model=128, n_heads=8)
x = torch.randn(2, 16, 128)
out = block(x)
print(f'output shape: {out.shape}')